# LeetCode 1576 — Replace All ?'s to Avoid Consecutive Repeating Characters bằng Python

**Link đề:** https://leetcode.com/problems/replace-all-s-to-avoid-consecutive-repeating-characters/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def modifyString(self, s: str) -> str:
        ...
```

### Mục tiêu học
1. Hiểu bài toán có **nhiều đáp án đúng** → phải kiểm thử bằng **hàm kiểm tra tính hợp lệ** (validator), không so khớp một chuỗi cố định.
2. Làm việc với chuỗi bất biến: đổi sang **list ký tự**, sửa tại chỗ, `"".join`.
3. Xử lý **biên trái/phải** an toàn (`i > 0`, `i < n - 1`) và nhớ so với ký tự bên trái **đã được thay**.
4. Nhận ra tư duy **tham lam** (greedy): mỗi `?` chỉ có tối đa 2 hàng xóm → chỉ cần 3 chữ cái `'a','b','c'` là luôn đủ (nguyên lý chuồng bồ câu).
5. Trace, chứng minh, Time/Space, test bằng validator + vét cạn, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — thay mọi dấu `?` sao cho không có hai ký tự kề nhau giống nhau

Cho chuỗi `s` gồm chữ cái thường và ký tự `'?'`. Hãy thay **mọi** `'?'` bằng chữ cái thường sao cho chuỗi cuối cùng **không có hai ký tự kề nhau giống nhau**. Không được sửa các ký tự khác `'?'`.

Đề **đảm bảo** chuỗi ban đầu không có hai ký tự (không phải `?`) kề nhau giống nhau. Nếu có nhiều đáp án, trả về **bất kỳ** đáp án nào.

| Ví dụ | `s` | Một output hợp lệ | Ghi chú |
|---|---|---|---|
| 1 | `"?zs"` | `"azs"` | 25 đáp án (`a..y` trừ `z`) |
| 2 | `"ubv?w"` | `"ubvaw"` | 24 đáp án (trừ `v`, `w`) |

**Constraints:** `1 <= s.length <= 100`; `s` chỉ gồm chữ cái thường và `'?'`.

Câu hỏi gợi mở:
- Với `"??"`, có thể thay thành `"ab"` — nhưng nếu thay từng dấu độc lập, chỉ nhìn chuỗi **gốc**, có thể ra `"aa"` không?
- Vì sao chỉ cần **3** chữ cái là đủ cho mọi trường hợp?

In [1]:
s = "ubv?w"
for i, ch in enumerate(s):
    left = s[i - 1] if i > 0 else None
    right = s[i + 1] if i < len(s) - 1 else None
    print(f"i={i}: '{ch}'  trái={left}  phải={right}")

i=0: 'u'  trái=None  phải=b
i=1: 'b'  trái=u  phải=v
i=2: 'v'  trái=b  phải=?
i=3: '?'  trái=v  phải=w
i=4: 'w'  trái=?  phải=None


Tại `i = 3` (dấu `?`), hàng xóm là `'v'` và `'w'` → chọn bất kỳ chữ nào khác hai chữ này.

## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi bất biến → list ký tự

Không gán được `s[i] = 'a'`. Ta đổi `chars = list(s)`, sửa `chars[i]`, rồi `"".join(chars)`.

In [2]:
s = "a?c"
chars = list(s)
chars[1] = "b"
print(chars, "→", "".join(chars))
try:
    s[1] = "b"
except TypeError as e:
    print("Không sửa chuỗi trực tiếp được:", e)

['a', 'b', 'c'] → abc
Không sửa chuỗi trực tiếp được: 'str' object does not support item assignment


### 2.2. Lấy hàng xóm an toàn ở biên

Ký tự đầu không có hàng xóm trái, ký tự cuối không có hàng xóm phải. Lưu ý: `s[i - 1]` với `i = 0` là `s[-1]` — **ký tự cuối** (không lỗi nhưng **sai nghĩa**)! Phải kiểm tra `i > 0`.

In [3]:
s = "?ab"
i = 0
print("s[i - 1] khi i=0:", repr(s[i - 1]), "← là ký tự CUỐI, không phải hàng xóm!")
left = s[i - 1] if i > 0 else ""
print("an toàn:", repr(left))

s[i - 1] khi i=0: 'b' ← là ký tự CUỐI, không phải hàng xóm!
an toàn: ''


### 2.3. Duyệt bảng chữ cái: `string.ascii_lowercase`, `chr`, `ord`

`string.ascii_lowercase == "abcdefghijklmnopqrstuvwxyz"`. Cũng có thể sinh bằng `chr(ord('a') + k)` với `k = 0..25`.

In [4]:
import string
print(string.ascii_lowercase)
print([chr(ord("a") + k) for k in range(5)])

abcdefghijklmnopqrstuvwxyz
['a', 'b', 'c', 'd', 'e']


### 2.4. `for ... break` và `in` với chuỗi/tuple

`c not in (left, right)` kiểm tra `c` khác cả hai. `break` thoát vòng ngay khi tìm được chữ phù hợp.

In [5]:
left, right = "v", "w"
for c in "uvwxyz":
    if c not in (left, right):
        print("chọn được:", c)
        break
    print("bỏ qua:", c)

chọn được: u


### 2.5. Hàm kiểm tra tính hợp lệ (validator)

Vì đáp án **không duy nhất**, ta viết hàm kiểm tra:
1. Độ dài giữ nguyên, ký tự không phải `?` giữ nguyên.
2. Không còn `?`, toàn chữ thường.
3. Không có hai ký tự kề nhau giống nhau.

In [6]:
def is_valid(original: str, result: str) -> bool:
    if len(original) != len(result):
        return False
    for o, r in zip(original, result):
        if o != "?" and o != r:          # sửa ký tự cố định → sai
            return False
        if not ("a" <= r <= "z"):          # còn '?' hoặc ký tự lạ → sai
            return False
    return all(result[i] != result[i + 1] for i in range(len(result) - 1))

print(is_valid("?zs", "azs"), is_valid("?zs", "zzs"), is_valid("?zs", "aza"), is_valid("??", "ab"))

True False False True


`"a" <= r <= "z"` so sánh chuỗi theo **thứ tự từ điển** (theo mã ký tự) — đúng cho một ký tự chữ thường.

## 3. Cách 1 — Thử lần lượt 26 chữ cái (trực giác)

Duyệt trái → phải trên list `chars`. Gặp `'?'` tại `i`: thử `c = 'a', 'b', ..., 'z'`, chọn chữ đầu tiên khác hàng xóm trái **(đã thay)** và hàng xóm phải.

In [7]:
def modify_try_26(s: str) -> str:
    chars = list(s)
    n = len(chars)
    for i in range(n):
        if chars[i] == "?":
            left = chars[i - 1] if i > 0 else ""
            right = chars[i + 1] if i < n - 1 else ""
            for c in string.ascii_lowercase:
                if c != left and c != right:
                    chars[i] = c
                    break
    return "".join(chars)

for s in ["?zs", "ubv?w", "??", "a?a", "????"]:
    r = modify_try_26(s)
    print(f"{s!r:>8} → {r!r:>8}  hợp lệ: {is_valid(s, r)}")

   '?zs' →    'azs'  hợp lệ: True
 'ubv?w' →  'ubvaw'  hợp lệ: True
    '??' →     'ab'  hợp lệ: True
   'a?a' →    'aba'  hợp lệ: True
  '????' →   'abab'  hợp lệ: True


**Điểm tinh tế:** `left = chars[i - 1]` đọc từ **list đang được sửa**, nên nếu bên trái là `?` đã được thay thì ta thấy chữ **mới**. Nếu đọc từ `s` gốc, `left` có thể là `'?'` — vô dụng (gợi ý thứ 2 của đề nhấn mạnh điều này).

Còn `right`: nếu bên phải cũng là `?` chưa thay, `right == '?'` — không trùng với chữ cái nào, nên không cản trở; khi tới lượt nó, nó sẽ né chữ ta vừa chọn.

## 4. Insight — chỉ cần 3 chữ cái

Mỗi `?` có **tối đa 2** hàng xóm → tối đa 2 chữ bị cấm. Trong **3** chữ `'a','b','c'` chắc chắn còn ít nhất một chữ không bị cấm (**nguyên lý chuồng bồ câu**: 2 "lệnh cấm" không thể chặn hết 3 lựa chọn).

Vậy thay vì thử 26 chữ, chỉ thử `"abc"` — vòng trong chạy tối đa 3 lần.

In [8]:
for left, right in [("a", "b"), ("b", "a"), ("a", "c"), ("c", "b"), ("", "a"), ("a", "a")]:
    pick = next(c for c in "abc" if c != left and c != right)
    print(f"trái={left!r:>4}, phải={right!r:>4} → chọn {pick!r}")

trái= 'a', phải= 'b' → chọn 'c'
trái= 'b', phải= 'a' → chọn 'c'
trái= 'a', phải= 'c' → chọn 'b'
trái= 'c', phải= 'b' → chọn 'a'
trái=  '', phải= 'a' → chọn 'b'
trái= 'a', phải= 'a' → chọn 'b'


`next(generator)` lấy phần tử **đầu tiên** thỏa điều kiện. Luôn tìm được vì lập luận trên (nếu không có, `next` sẽ báo `StopIteration`).

## 5. Lời giải chính — Tham lam với `"abc"`

### 5.1. Từng đoạn
1. `chars = list(s)`, `n = len(chars)`.
2. Với mỗi `i` mà `chars[i] == '?'`:
   - `left = chars[i-1]` nếu `i > 0`, ngược lại `''`.
   - `right = chars[i+1]` nếu `i < n-1`, ngược lại `''`.
   - Chọn `c` đầu tiên trong `"abc"` khác `left` và `right`; gán `chars[i] = c`.
3. `return "".join(chars)`.

### 5.2. Ghép lại

In [9]:
def modify_abc(s: str) -> str:
    chars = list(s)
    n = len(chars)
    for i in range(n):
        if chars[i] != "?":
            continue
        left = chars[i - 1] if i > 0 else ""
        right = chars[i + 1] if i < n - 1 else ""
        for c in "abc":
            if c != left and c != right:
                chars[i] = c
                break
    return "".join(chars)

for s in ["?zs", "ubv?w", "??", "a?a", "????", "b?a", "?"]:
    r = modify_abc(s)
    print(f"{s!r:>8} → {r!r:>8}  hợp lệ: {is_valid(s, r)}")

   '?zs' →    'azs'  hợp lệ: True
 'ubv?w' →  'ubvaw'  hợp lệ: True
    '??' →     'ab'  hợp lệ: True
   'a?a' →    'aba'  hợp lệ: True
  '????' →   'abab'  hợp lệ: True
   'b?a' →    'bca'  hợp lệ: True
     '?' →      'a'  hợp lệ: True


## 6. TRACE từng bước

`s = "b??a?"`:

| i | ký tự | left (đã thay) | right | thử `a` | thử `b` | thử `c` | chọn | chars sau |
|---|---|---|---|---|---|---|---|---|
| 0 | b | — | — | | | | (giữ) | `b??a?` |
| 1 | ? | b | ? | ✓ | | | a | `ba?a?` |
| 2 | ? | a | a | ✗ | ✓ | | b | `baba?` |
| 3 | a | — | — | | | | (giữ) | `baba?` |
| 4 | ? | a | (hết) | ✗ | ✓ | | b | `babab` |

In [10]:
def trace_modify(s):
    chars = list(s); n = len(chars)
    print(f"s = {s!r}")
    for i in range(n):
        if chars[i] != "?":
            continue
        left = chars[i - 1] if i > 0 else ""
        right = chars[i + 1] if i < n - 1 else ""
        tried = []
        for c in "abc":
            tried.append(c)
            if c != left and c != right:
                chars[i] = c
                break
        print(f"i={i}: left={left!r:>4}, right={right!r:>4}, thử {tried} → chọn {chars[i]!r} | {''.join(chars)}")
    return "".join(chars)

r = trace_modify("b??a?")
assert r == "babab" and is_valid("b??a?", r)
print()
r = trace_modify("??c??")
assert is_valid("??c??", r)

s = 'b??a?'
i=1: left= 'b', right= '?', thử ['a'] → chọn 'a' | ba?a?
i=2: left= 'a', right= 'a', thử ['a', 'b'] → chọn 'b' | baba?
i=4: left= 'a', right=  '', thử ['a', 'b'] → chọn 'b' | babab

s = '??c??'
i=0: left=  '', right= '?', thử ['a'] → chọn 'a' | a?c??
i=1: left= 'a', right= 'c', thử ['a', 'b'] → chọn 'b' | abc??
i=3: left= 'c', right= '?', thử ['a'] → chọn 'a' | abca?
i=4: left= 'a', right=  '', thử ['a', 'b'] → chọn 'b' | abcab


Ở trace thứ hai, `i = 1`: `left = 'a'` (vừa thay ở `i = 0`), `right = 'c'` → thử `a` ✗, `b` ✓. Nếu ta lấy `left` từ chuỗi gốc (`'?'`), sẽ chọn `'a'` → `"aac.."` sai!

## 7. Các lời giải khác

### 7.1. Cách 3 — Tập chữ bị cấm (`set`)

Thay vì thử lần lượt, ta gom các chữ bị cấm (hàng xóm trái/phải) vào một `set`, rồi lấy chữ nhỏ nhất trong `{a, b, c}` **không** thuộc tập cấm. (Một ý tưởng hay bị nghĩ tới là gán cứng `"abc"[i % 3]` — nhưng có thể trùng với chữ cố định bên cạnh, nên vẫn phải kiểm tra; cách tập cấm gọn hơn.)

In [11]:
def modify_set(s: str) -> str:
    chars = list(s)
    n = len(chars)
    for i, ch in enumerate(chars):
        if ch == "?":
            banned = set()
            if i > 0:
                banned.add(chars[i - 1])
            if i < n - 1:
                banned.add(chars[i + 1])
            chars[i] = min(set("abc") - banned)
    return "".join(chars)

for s in ["?zs", "ubv?w", "b??a?"]:
    r = modify_set(s)
    print(s, "→", r, is_valid(s, r))

?zs → azs True
ubv?w → ubvaw True
b??a? → babab True


`set("abc") - banned` là **hiệu tập**: các chữ trong `{a, b, c}` không bị cấm; `min(...)` chọn chữ nhỏ nhất (theo bảng chữ cái) để kết quả **xác định** (deterministic).

Chú ý: `enumerate(chars)` vẫn đúng dù ta sửa `chars[i]` trong lúc duyệt, vì ta chỉ **gán** (không thêm/xóa phần tử) và chỉ sửa ô hiện tại.

### 7.2. Cách 4 — Vét cạn (chỉ để kiểm chứng)

Thử **mọi** cách thay bằng `itertools.product` (với bảng chữ `"abc"`), lấy cách hợp lệ đầu tiên. Độ phức tạp `3^q` (q = số dấu `?`) — chỉ dùng cho chuỗi ngắn để xác nhận "luôn có đáp án" và đối chiếu.

In [12]:
from itertools import product

def modify_bruteforce(s: str, alphabet: str = "abc"):
    qs = [i for i, ch in enumerate(s) if ch == "?"]
    for combo in product(alphabet, repeat=len(qs)):
        chars = list(s)
        for idx, c in zip(qs, combo):
            chars[idx] = c
        cand = "".join(chars)
        if is_valid(s, cand):
            return cand
    return None

print(list(product("ab", repeat=2)))
print(modify_bruteforce("b??a?"), modify_bruteforce("?zs"))

[('a', 'a'), ('a', 'b'), ('b', 'a'), ('b', 'b')]
babab azs


`product("ab", repeat=2)` sinh mọi bộ 2 phần tử từ `"ab"`: `('a','a'), ('a','b'), ('b','a'), ('b','b')` — tương đương 2 vòng lặp lồng.

### 7.3. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Thử 26 chữ | chữ đầu tiên khác 2 hàng xóm | `O(26·n)` = `O(n)` | `O(n)` | trực giác |
| 2. **Tham lam `"abc"`** | 3 chữ là đủ | **`O(n)`** (≤ 3 lần thử/ký tự) | `O(n)` | **nên dùng** |
| 3. Tập cấm | `min({a,b,c} - banned)` | `O(n)` | `O(n)` | gọn, xác định |
| 4. Vét cạn | `product` mọi cách thay | `O(3^q · n)` | `O(n)` | chỉ để kiểm chứng |

`O(n)` space là cho list `chars` / chuỗi kết quả — bắt buộc vì phải trả về chuỗi mới.

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def modifyString(self, s: str) -> str:
        chars = list(s)
        n = len(chars)
        for i in range(n):
            if chars[i] == "?":
                left = chars[i - 1] if i > 0 else ""
                right = chars[i + 1] if i < n - 1 else ""
                for c in "abc":
                    if c != left and c != right:
                        chars[i] = c
                        break
        return "".join(chars)

sol = Solution()
print(sol.modifyString("?zs"))
print(sol.modifyString("ubv?w"))

azs
ubvaw


## 9. Vì sao đúng?

**Luôn chọn được:** tại mỗi `?`, tập chữ bị cấm gồm tối đa 2 phần tử (`left`, `right`; `right` có thể là `'?'` — không phải chữ cái nên không cấm gì). Ba ứng viên `a, b, c` → luôn còn ít nhất một ✓.

**Invariant:** sau khi xử lý xong vị trí `i`, tiền tố `chars[0..i]` không còn `?` và **không có cặp kề nhau giống nhau**.
- Nếu `chars[i]` là chữ cố định: cặp `(chars[i-1], chars[i])` — nếu `chars[i-1]` là chữ cố định thì đề đảm bảo khác nhau; nếu `chars[i-1]` là `?` đã thay, lúc thay nó đã né `right = chars[i]` ✓.
- Nếu `chars[i]` là `?`: ta chọn `c != left` (chữ đã chốt bên trái) ✓.

**Không phá phần bên phải:** chữ chọn cho `?` cũng khác `right` nếu `right` là chữ cố định. Nếu `right` là `?`, nó sẽ được xử lý sau và tự né `c`.

Kết thúc `i = n-1`: toàn chuỗi hợp lệ; ký tự cố định không bị đụng vào (chỉ gán khi `chars[i] == '?'`) ✓.

## 10. Time / Space

Gọi `n = len(s)`.

- **Time:** mỗi vị trí làm `O(1)` việc (tối đa 3 lần thử) → **`O(n)`**. Bản thử 26 chữ cũng `O(n)` nhưng hằng số lớn hơn.
- **Space:** list `chars` + chuỗi kết quả → **`O(n)`**. List `chars` là cần thiết vì chuỗi Python bất biến; ngoài nó ra chỉ dùng vài biến `O(1)` (`left`, `right`, `c`).

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Vì đáp án không duy nhất, test bằng `is_valid` thay vì `==` (trừ khi biết chắc hàm trả chuỗi cụ thể).

In [14]:
def check_valid(name, s):
    r = Solution().modifyString(s)
    assert is_valid(s, r), (name, s, r)
    print(f"PASS {name}: {s!r} → {r!r}")

check_valid("đề 1", "?zs")
check_valid("đề 2", "ubv?w")
check_valid("một dấu ?", "?")
check_valid("không có ?", "abcdef")
check_valid("toàn ?", "?" * 7)
check_valid("? ở cuối", "ab?")
check_valid("? kẹp a..b", "a?b")
check_valid("? kẹp a..a", "a?a")
check_valid("chuỗi ? cạnh a và b", "a???b")
check_valid("xen kẽ", "?a?b?c?")
check_valid("dài 100", ("?ab" * 34)[:100])

assert Solution().modifyString("?zs") == "azs"          # hàm của ta trả đúng như ví dụ đề
assert Solution().modifyString("ubv?w") == "ubvaw"

PASS đề 1: '?zs' → 'azs'
PASS đề 2: 'ubv?w' → 'ubvaw'
PASS một dấu ?: '?' → 'a'
PASS không có ?: 'abcdef' → 'abcdef'
PASS toàn ?: '???????' → 'abababa'
PASS ? ở cuối: 'ab?' → 'aba'
PASS ? kẹp a..b: 'a?b' → 'acb'
PASS ? kẹp a..a: 'a?a' → 'aba'
PASS chuỗi ? cạnh a và b: 'a???b' → 'abacb'
PASS xen kẽ: '?a?b?c?' → 'bacbaca'
PASS dài 100: '?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?ab?' → 'babcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcabcaba'


**Phản ví dụ — đọc hàng xóm trái từ chuỗi gốc:**

In [15]:
def modify_read_original_bug(s):
    chars = list(s); n = len(s)
    for i in range(n):
        if s[i] == "?":
            left = s[i - 1] if i > 0 else ""          # SAI: đọc s gốc
            right = s[i + 1] if i < n - 1 else ""
            for c in "abc":
                if c != left and c != right:
                    chars[i] = c
                    break
    return "".join(chars)

for s in ["??", "????", "a??"]:
    r = modify_read_original_bug(s)
    print(f"{s!r}: lỗi → {r!r} (hợp lệ? {is_valid(s, r)}) | đúng → {modify_abc(s)!r}")

'??': lỗi → 'aa' (hợp lệ? False) | đúng → 'ab'
'????': lỗi → 'aaaa' (hợp lệ? False) | đúng → 'abab'
'a??': lỗi → 'aba' (hợp lệ? True) | đúng → 'aba'


### 11.1. Đối chiếu trên chuỗi ngẫu nhiên + kiểm chứng "luôn có đáp án"

Sinh chuỗi thỏa cam kết của đề (không có hai chữ cố định kề nhau giống nhau). Kiểm tra mọi cách đều hợp lệ, và vét cạn (bảng `"abc"`) luôn tìm thấy đáp án.

In [16]:
import random
rng = random.Random(1576)

def random_input(n):
    out = []
    for _ in range(n):
        while True:
            ch = "?" if rng.random() < 0.5 else rng.choice("abcz")
            if ch == "?" or not out or out[-1] != ch:
                out.append(ch); break
    return "".join(out)

for _ in range(3000):
    s = random_input(rng.randint(1, 12))
    for f in (modify_try_26, modify_abc, modify_set, Solution().modifyString):
        assert is_valid(s, f(s)), (f, s)
    assert modify_abc(s) == modify_set(s)             # cùng quy tắc "chữ nhỏ nhất" → cùng kết quả
    if s.count("?") <= 6:
        assert modify_bruteforce(s) is not None
print("3000 chuỗi ngẫu nhiên: mọi cách đều hợp lệ; vét cạn luôn có đáp án")

3000 chuỗi ngẫu nhiên: mọi cách đều hợp lệ; vét cạn luôn có đáp án


### 11.2. Hai chữ cái có đủ không? — phản ví dụ cho "tham lam với `ab`"

Nếu chỉ dùng `"ab"`, `?` kẹp giữa `a` và `b` không có lựa chọn nào → bắt buộc cần chữ thứ ba.

In [17]:
print("Vét cạn với 'ab' cho 'a?b':", modify_bruteforce("a?b", alphabet="ab"))
print("Vét cạn với 'abc' cho 'a?b':", modify_bruteforce("a?b", alphabet="abc"))

Vét cạn với 'ab' cho 'a?b': None
Vét cạn với 'abc' cho 'a?b': acb


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Đọc hàng xóm trái từ chuỗi gốc
Đã thấy ở mục 11: `"??"` → `"aa"`. **Sửa:** đọc từ `chars` (list đang được cập nhật).

### Lỗi 2 — `s[i - 1]` khi `i = 0`
Python cho `s[-1]` là ký tự **cuối** → né nhầm chữ, có khi vô hại, có khi làm sai logic khó tìm. **Sửa:** `if i > 0`.

In [18]:
def modify_negative_index_bug(s):
    chars = list(s); n = len(s)
    for i in range(n):
        if chars[i] == "?":
            left = chars[i - 1]                       # SAI khi i = 0
            right = chars[i + 1] if i < n - 1 else ""
            for c in "ab":                            # chỉ 2 chữ để lỗi lộ ra
                if c != left and c != right:
                    chars[i] = c; break
    return "".join(chars)

s = "?ba"
r = modify_negative_index_bug(s)
print(f"{s!r} → {r!r}, hợp lệ? {is_valid(s, r)}  (left của i=0 bị lấy nhầm là 'a' cuối chuỗi)")

'?ba' → '?ba', hợp lệ? False  (left của i=0 bị lấy nhầm là 'a' cuối chuỗi)


Ở đây `left = 'a'` (ký tự cuối), `right = 'b'` → không chọn được chữ nào trong `"ab"` → `?` **không được thay**. Với `"abc"` lỗi bị che khuất, nhưng vẫn là bug.

### Lỗi 3 — `chars[i + 1]` khi `i = n - 1`
`IndexError` khi `?` ở cuối. **Sửa:** `if i < n - 1`.

### Lỗi 4 — Thay bằng một chữ cố định
`s.replace("?", "a")` — sai ngay với `"a?"` hoặc `"??"`.

In [19]:
for s in ["a?", "??", "?b"]:
    r = s.replace("?", "a")
    print(f"{s!r}.replace → {r!r}, hợp lệ? {is_valid(s, r)}")

'a?'.replace → 'aa', hợp lệ? False
'??'.replace → 'aa', hợp lệ? False
'?b'.replace → 'ab', hợp lệ? True


### Lỗi 5 — Kiểm thử bằng `==` với một đáp án cụ thể
Code đúng nhưng chọn chữ khác (ví dụ `"bzs"`) vẫn hợp lệ. Test cứng `== "azs"` chỉ đúng với **một** cách cài đặt. Dùng validator.

## 13. Mẹo nhận dạng pattern

- "Thay chỗ trống sao cho không có hai phần tử kề nhau giống nhau" → **tham lam trái → phải**, mỗi vị trí chỉ phụ thuộc hàng xóm.
- Mỗi vị trí có `d` ràng buộc "khác" → chỉ cần `d + 1` lựa chọn (**chuồng bồ câu**). Hàng xóm 2 phía → 3 màu. Đây là ý tưởng **tô màu đồ thị tham lam**: đồ thị bậc tối đa `Δ` luôn tô được bằng `Δ + 1` màu.
- Đề nói "return any of them" → test bằng **validator** và/hoặc **vét cạn trên input nhỏ**.
- Khi sửa list tại chỗ, phân biệt rõ đọc **giá trị gốc** hay **giá trị đã cập nhật**.
- Bài liên quan: 1784, 984 *String Without AAA or BBB*, 767 *Reorganize String*, 1405 *Longest Happy String*.

## 14. Biến thể

### 14.1. Không được có 3 ký tự kề nhau giống nhau (cho phép 2)
Mỗi `?` cần khác **cặp** trái hoặc **cặp** phải khi cặp đó gồm hai chữ giống nhau. Vẫn tham lam được nhưng điều kiện phức tạp hơn; xem bài tập 6.

### 14.2. Cạnh vòng tròn: ký tự đầu và cuối cũng coi là kề nhau
Hàng xóm của `i = 0` gồm `chars[1]` và `chars[n-1]`. Với `n >= 3` vẫn chỉ 2 hàng xóm → 3 chữ đủ. Nhưng vì `?` ở đầu chọn trước khi `?` ở cuối được thay, ta cần cập nhật đúng thứ tự.

In [20]:
def modify_circular(s: str) -> str:
    chars = list(s); n = len(chars)
    for i in range(n):
        if chars[i] == "?":
            banned = {chars[(i - 1) % n], chars[(i + 1) % n]} if n > 1 else set()
            chars[i] = min(set("abc") - banned)
    return "".join(chars)

def is_valid_circular(original, result):
    n = len(result)
    return is_valid(original, result) and (n == 1 or result[0] != result[-1])

for s in ["???", "a??", "?b?a", "?"]:
    r = modify_circular(s)
    print(s, "→", r, is_valid_circular(s, r))

??? → abc True
a?? → abc True
?b?a → cbca True
? → a True


`(i - 1) % n` cho hàng xóm vòng tròn: với `i = 0` là `n - 1`. Lưu ý: nếu chuỗi gốc có `s[0] == s[-1]` là chữ cố định thì vô nghiệm — ở đây ta giả sử input hợp lệ.

### 14.3. Đếm số cách thay hợp lệ
Mỗi đoạn `?` liên tiếp độc lập với nhau → dùng **quy hoạch động** nhỏ: `dp[c]` = số cách sao cho ký tự hiện tại là `c`.

In [21]:
def count_ways(s: str, alphabet: str = string.ascii_lowercase) -> int:
    dp = {c: 1 for c in alphabet} if s[0] == "?" else {s[0]: 1}
    for ch in s[1:]:
        options = alphabet if ch == "?" else ch
        total_prev = sum(dp.values())
        dp = {c: total_prev - dp.get(c, 0) for c in options}
    return sum(dp.values())

print(count_ways("?zs"), count_ways("ubv?w"))       # 25, 24 — khớp với đề
print(count_ways("??", "abc"), sum(1 for a, b2 in product("abc", repeat=2) if a != b2))

25 24
6 6


Kết quả `25` và `24` khớp **chính xác** với lời giải thích trong đề — một cách tự kiểm tra rất thuyết phục.

## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
Chạy tham lam `"abc"` trên `"?a??b?"`. Ghi `left`, `right`, chữ chọn ở mỗi `?`.

### Bài 2 — Đoán output
```python
s = "?x"
i = 0
print(repr(s[i - 1]), min(set("abc") - {"a", "x"}))
```

### Bài 3 — TODO: viết lời giải tham lam

In [22]:
def exercise_modify_string(s: str) -> str:
    # TODO 1: chars = list(s)
    # TODO 2: với mỗi '?', lấy left (đã thay) / right an toàn ở biên
    # TODO 3: chọn chữ đầu tiên trong "abc" khác left và right
    pass

# r = exercise_modify_string("ubv?w"); assert is_valid("ubv?w", r)
# r = exercise_modify_string("????"); assert is_valid("????", r)

### Bài 4 — TODO: viết `is_valid`
Tự viết hàm kiểm tra như mục 2.5 (không xem lại!).

In [23]:
def exercise_is_valid(original: str, result: str) -> bool:
    # TODO: độ dài, ký tự cố định giữ nguyên, không còn '?', không có cặp kề giống nhau
    pass

# assert exercise_is_valid("?zs", "azs") is True
# assert exercise_is_valid("?zs", "zzs") is False
# assert exercise_is_valid("?zs", "azz") is False

### Bài 5 — TODO: kết quả nhỏ nhất theo thứ tự từ điển
Trả về chuỗi hợp lệ **nhỏ nhất theo thứ tự từ điển**. (Tham lam `"abc"` có cho đúng không? Hãy lập luận rồi đối chiếu với vét cạn.)

In [24]:
def exercise_lexicographically_smallest(s: str) -> str:
    # TODO
    pass

# assert exercise_lexicographically_smallest("??") == "ab"
# assert exercise_lexicographically_smallest("?a?") == "bab"

### Bài 6 — Mini challenge: cấm 3 ký tự giống nhau liên tiếp
Thay mọi `?` sao cho **không có 3 ký tự liên tiếp giống nhau** (2 thì được). Giả sử chuỗi gốc chưa có 3 chữ cố định liên tiếp giống nhau. Ưu tiên dùng `'a'` nhiều nhất có thể.

In [25]:
def exercise_no_three_in_row(s: str) -> str:
    # TODO: tại '?', cấm c nếu (2 ký tự trái đều là c) hoặc (2 ký tự phải đều là c) hoặc (trái là c và phải là c)
    pass

# r = exercise_no_three_in_row("aa?aa"); assert "aaa" not in r

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đi trái → phải. (2) `left` lấy từ chữ đã chốt. (3) Thử `a`, `b`, `c` theo thứ tự.

**Bài 2:** (1) Chỉ số âm. (2) Hiệu tập. (3) `'x'` và `'b'`.

**Bài 3:** (1) Như mục 5. (2) `if i > 0`, `if i < n - 1`. (3) `for c in "abc": ... break`.

**Bài 4:** (1) `zip(original, result)`. (2) `o != "?" and o != r` → sai. (3) `all(result[i] != result[i+1] ...)`.

**Bài 5:** (1) Đặt chữ nhỏ nhất có thể ở vị trí sớm nhất là tốt nhất cho thứ tự từ điển. (2) Chữ nhỏ nhất hợp lệ luôn nằm trong `a, b, c`. (3) Tham lam `"abc"` **chính là** đáp án nhỏ nhất — kiểm chứng bằng vét cạn với bảng 26 chữ trên chuỗi ngắn.

**Bài 6:** (1) Điều kiện cấm dựa trên **cặp** ký tự. (2) Ba trường hợp: `chars[i-2] == chars[i-1] == c`, `chars[i+1] == chars[i+2] == c` (chỉ khi là chữ cố định), `chars[i-1] == chars[i+1] == c`. (3) Thử `"abc"`, chữ đầu tiên không bị cấm.

## 17. Đáp án tham khảo

**Bài 1:** `"?a??b?"`: `i=0` (left `''`, right `a`) → `b`; `i=2` (left `a`, right `?`) → `b`; `i=3` (left `b`, right `b`) → `a`; `i=5` (left `b`, right `''`) → `a`. Kết quả `"bababa"`.

**Bài 2:** `'x' b`.

In [26]:
def answer_modify_string(s):
    chars = list(s); n = len(chars)
    for i in range(n):
        if chars[i] == "?":
            left = chars[i - 1] if i > 0 else ""
            right = chars[i + 1] if i < n - 1 else ""
            for c in "abc":
                if c != left and c != right:
                    chars[i] = c; break
    return "".join(chars)

def answer_is_valid(original, result):
    if len(original) != len(result):
        return False
    if any(o != "?" and o != r for o, r in zip(original, result)):
        return False
    if any(not ("a" <= r <= "z") for r in result):
        return False
    return all(result[i] != result[i + 1] for i in range(len(result) - 1))

def answer_lexicographically_smallest(s):
    return answer_modify_string(s)          # tham lam "abc" đã cho chuỗi nhỏ nhất

def answer_no_three_in_row(s):
    chars = list(s); n = len(chars)
    for i in range(n):
        if chars[i] != "?":
            continue
        for c in "abc":
            bad = (i >= 2 and chars[i - 2] == chars[i - 1] == c) \
                or (i + 2 < n and chars[i + 1] == chars[i + 2] == c) \
                or (0 < i < n - 1 and chars[i - 1] == c == chars[i + 1])
            if not bad:
                chars[i] = c; break
    return "".join(chars)

assert answer_modify_string("?a??b?") == "bababa"
s = "?x"; assert (repr(s[0 - 1]), min(set("abc") - {"a", "x"})) == ("'x'", "b")
assert answer_is_valid("?zs", "azs") is True
assert answer_is_valid("?zs", "zzs") is False
assert answer_is_valid("?zs", "azz") is False
assert answer_lexicographically_smallest("??") == "ab"
assert answer_lexicographically_smallest("?a?") == "bab"
r = answer_no_three_in_row("aa?aa"); assert "aaa" not in r, r
rng = random.Random(76)
for _ in range(400):
    s = random_input(rng.randint(1, 5))
    # bài 5: so với vét cạn trên cả bảng 26 chữ (vét cạn sinh theo thứ tự từ điển → đáp án đầu tiên là nhỏ nhất)
    assert answer_lexicographically_smallest(s) == modify_bruteforce(s, alphabet=string.ascii_lowercase)
    assert answer_is_valid(s, answer_modify_string(s)) == is_valid(s, answer_modify_string(s)) == True
for _ in range(400):
    raw = "".join(rng.choice("aa?") for _ in range(rng.randint(1, 10)))
    if "aaa" in raw.replace("?", "b"):
        continue                              # input vi phạm giả thiết
    r = answer_no_three_in_row(raw)
    assert "?" not in r and all(not (r[i] == r[i + 1] == r[i + 2]) for i in range(len(r) - 2)), (raw, r)
    assert all(o == "?" or o == x for o, x in zip(raw, r))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


Kiểm chứng bài 5 dựa vào việc `product` sinh tổ hợp theo **thứ tự từ điển**, nên đáp án hợp lệ **đầu tiên** của vét cạn chính là chuỗi nhỏ nhất.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def modifyString(self, s: str) -> str:
        chars, n = list(s), len(s)
        for i in range(n):
            if chars[i] == "?":
                left = chars[i - 1] if i > 0 else ""          # đọc chữ ĐÃ thay
                right = chars[i + 1] if i < n - 1 else ""
                for c in "abc":                               # 2 hàng xóm → 3 chữ là đủ
                    if c != left and c != right:
                        chars[i] = c
                        break
        return "".join(chars)
```

**Câu thần chú:** *"Hai hàng xóm cấm tối đa hai chữ — a, b, c luôn còn một."*

**Checklist trước khi submit**
- [ ] Đọc `left` từ list **đã cập nhật**, không từ `s` gốc.
- [ ] Bảo vệ biên: `i > 0` (tránh `s[-1]`), `i < n - 1` (tránh `IndexError`).
- [ ] Không sửa ký tự khác `?`.
- [ ] Test `"?"`, `"??"`, `"a?b"`, `?` ở đầu/cuối; kiểm tra bằng validator.
- [ ] Đúng tên method `modifyString`, trả về `str`.